## Naïve RAG (PDF)
This will be built upon Docling's research paper. Will be used for answering the questions based on that paper.

In [1]:
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pathlib import Path
from pydantic import BaseModel
import json

import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from openai import OpenAI
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)
from deepeval import evaluate
from deepeval.metrics import BaseMetric
from rag_eval import evaluate_dataset

In [2]:
load_dotenv(override=True)

True

In [3]:
openai = OpenAI()

### Document Loading (Parsing)
Docling's research paper can downloaded from [here](https://arxiv.org/pdf/2408.09869).

In [4]:
file_path = list(Path.cwd().rglob("docling_paper.pdf"))
doc = pymupdf4llm.to_markdown(str(file_path[0]), use_ocr=False)

In [ ]:
display(Markdown(doc))

### Chunking

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
all_splits = text_splitter.split_text(doc)

In [7]:
all_splits

[Document(metadata={}, page_content='**==> picture [100 x 99] intentionally omitted <==**'),
 Document(metadata={'Header 2': '**Docling Technical Report**'}, page_content='Version 1.0  \n**Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar**  \nAI4K Group, IBM Research R¨uschlikon, Switzerland'),
 Document(metadata={'Header 2': '**Abstract**'}, page_content='This technical report introduces _Docling_ , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility a

In [8]:
# We can reject the first and last one i.e. appendix, since
# it contains images and their OCR is just too bad.
splits = all_splits[1:-1]

### Embedding

In [9]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [10]:
e = embeddings.embed_query("Hello, world!")

### Storing it in Vector Database

In [11]:
id = [f"chunk_{i}" for i in range(len(splits))]

In [12]:
db_name = Path.cwd() / "vdb"

if db_name.exists():
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_store = Chroma.from_documents(
    documents=splits, embedding=embeddings, persist_directory=str(db_name), ids=id
)
print(f"Vectorstore created with {vector_store._collection.count()} documents")

Vectorstore created with 16 documents


In [13]:
collection = vector_store._collection
result = collection.get()

In [14]:
result

{'ids': ['chunk_0',
  'chunk_1',
  'chunk_2',
  'chunk_3',
  'chunk_4',
  'chunk_5',
  'chunk_6',
  'chunk_7',
  'chunk_8',
  'chunk_9',
  'chunk_10',
  'chunk_11',
  'chunk_12',
  'chunk_13',
  'chunk_14',
  'chunk_15'],
 'embeddings': None,
 'documents': ['Version 1.0  \n**Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar**  \nAI4K Group, IBM Research R¨uschlikon, Switzerland',
  'This technical report introduces _Docling_ , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allow

### Retrieving

In [15]:
result = vector_store.similarity_search(
    query="who are the authors of this paper and what is ocr?", k=2
)

In [16]:
result

[Document(id='chunk_9', metadata={'Header 2': '**OCR**'}, page_content='Docling provides optional support for OCR, for example to cover scanned PDFs or content in bitmaps images embedded on a page. In our initial release, we rely on _EasyOCR_ [1], a popular thirdparty OCR library with support for many languages. Docling, by default, feeds a high-resolution page image (216 dpi) to the OCR engine, to allow capturing small print detail in decent quality. While EasyOCR delivers reasonable transcription quality, we observe that it runs fairly slow on CPU (upwards of 30 seconds per page).  \nWe are actively seeking collaboration from the open-source community to extend Docling with additional OCR backends and speed improvements.'),
 Document(id='chunk_15', metadata={'Header 2': '**References**'}, page_content='- [1] J. AI. Easyocr: Ready-to-use ocr with 80+ supported languages. `https://github.com/ JaidedAI/EasyOCR` , 2024. Version: 1.7.0.  \n- [2] J. Ansel, E. Yang, H. He, N. Gimelshein, A.

In [17]:
def context_building(context: str) -> str:
    extended_prompt = f"""You are an smart AI assistant helping for reading a research paper. You will answer the question in explainatory way.
	If relevant, use the given context to answer any question.
	If you don't know the answer, say so.
	Context:
	{context}"""
    return extended_prompt

In [18]:
def rag_pipeline(query: str) -> str:
    retriever = vector_store.similarity_search(query=query, k=1)
    context = ["\n\n" + i.page_content for i in retriever]
    extended_prompt = context_building(context)
    result = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": extended_prompt},
            {"role": "user", "content": query},
        ],
    )
    result = result.choices[0].message.content
    data = {
        "question": query,
        "answer": result,
        "contexts": [i.page_content for i in retriever],
        "chunk_ids": [i.id for i in retriever],
    }
    return data

In [19]:
rag_pipeline("who are the authors of this paper")

{'question': 'who are the authors of this paper',
 'answer': 'The authors of the paper are:\n\n- Christoph Auer\n- Maksym Lysak\n- Ahmed Nassar\n- Michele Dolfi\n- Nikolaos Livathinos\n- Panos Vagenas\n- Cesar Berrospi Ramis\n- Matteo Omenetti\n- Fabian Lindlbauer\n- Kasper Dinkla\n- Lokesh Mishra\n- Yusik Kim\n- Shubham Gupta\n- Rafael Teixeira de Lima\n- Valery Weber\n- Lucas Morin\n- Ingmar Meijer\n- Viktor Kuropiatnyk\n- Peter W. J. Staar\n\nThey are all associated with the AI4K Group at IBM Research in Rüschlikon, Switzerland.',
 'contexts': ['Version 1.0  \n**Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar**  \nAI4K Group, IBM Research R¨uschlikon, Switzerland'],
 'chunk_ids': ['chunk_0']}

### Eval

#### Creating golden dataset

To generate the golden dataset, we can do it by 3 ways,
1. Manually create it (mehh)
2. Pass your chunks to LLM and generates questions from it
3. Use Ragas to automatically generate it

We'll be using 2nd way, as ragas is not working as of now. 

In [20]:
class CoreFormat(BaseModel):
    question: str
    answer: str

In [21]:
gd_path = str(Path.cwd() / "eval" / "golden_dataset.jsonl")

for i, chunk in enumerate(splits):
    chunk_id = f"chunk_{i}"

    messages = [
        {
            "role": "system",
            "content": "Generate a question that requires understanding this text, but do NOT copy phrases directly. Also answer",
        },
        {"role": "user", "content": chunk.page_content},
    ]

    response = openai.chat.completions.parse(
        model="gpt-4.1-nano",
        messages=messages,
        response_format=CoreFormat,
    )

    # Convert pydantic model to dict for JSON serialization (use model_dump instead of dict)
    qa = response.choices[0].message.parsed.model_dump()

    gd = {
        "question": qa["question"],
        "ground_truth": qa["answer"],
        "contexts": [chunk.page_content],
        "golden_chunk_id": [chunk_id],
    }

    with open(gd_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(gd, ensure_ascii=False) + "\n")

#### Using Ragas for evaluation

There is an ongoing issue with Ragas, so not able to use it as of now, below are the some open issues,
- https://github.com/vibrantlabsai/ragas/issues/2753
- https://github.com/vibrantlabsai/ragas/issues/2745
- https://github.com/vibrantlabsai/ragas/issues/2741

#### Using DeepEval for evaluation

In [22]:
test_cases_path = str(Path.cwd() / "eval" / "test_cases.jsonl")

test_cases = []

with open(gd_path) as f:
    for line in f:
        row = json.loads(line)

        result = rag_pipeline(row["question"])

        # ------- Writing chunks in a seprate file -------
        chunks = {
            "question": row["question"],
            "golden_chunk_ids": row["golden_chunk_id"],
            "retrieved_chunk_ids": result["chunk_ids"],
        }

        with open(test_cases_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(chunks, ensure_ascii=False) + "\n")

        # ------- End -------

        test_cases.append(
            LLMTestCase(
                input=row["question"],
                actual_output=result["answer"],
                expected_output=row["ground_truth"],
                retrieval_context=result["contexts"],
                metadata={
                    "golden_chunk_ids": row["golden_chunk_id"],
                    "retrieved_chunk_ids": result["chunk_ids"],
                },
            )
        )

In [23]:
# This is an optional block, since we'll be calcualting these matrices without
# DeepEvals (from my own package)

# Hit@k
class RetrievalHitRate(BaseMetric):
    def __init__(self, threshold=1.0):
        self.threshold = threshold
        self.score = 0.0
        self.reason = ""

    def measure(self, test_case):
        metadata = test_case.metadata or {}

        golden = set(metadata.get("golden_chunk_ids", []))
        retrieved = set(metadata.get("retrieved_chunk_ids", []))

        hit = len(golden & retrieved) > 0
        self.score = 1.0 if hit else 0.0

        self.reason = f"golden={golden}, retrieved={retrieved}"
        return self.score

    async def a_measure(self, test_case):
        return self.measure(test_case)

    def is_successful(self):
        return self.score >= self.threshold

    @property
    def name(self):
        return "Retrieval Hit Rate"


# Recall@k
class RetrievalRecall(BaseMetric):
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.score = 0.0
        self.reason = ""

    def measure(self, test_case):
        metadata = test_case.metadata or {}

        golden = set(metadata.get("golden_chunk_ids", []))
        retrieved = set(metadata.get("retrieved_chunk_ids", []))

        if len(golden) == 0:
            self.score = 0.0
            self.reason = "No golden chunk ids"
            return self.score

        self.score = len(golden & retrieved) / len(golden)
        self.reason = f"golden={golden}, retrieved={retrieved}"

        return self.score

    async def a_measure(self, test_case):
        return self.measure(test_case)

    def is_successful(self):
        return self.score >= self.threshold

    @property
    def name(self):
        return "Retrieval Recall"

In [24]:
metrics = [
    RetrievalHitRate(),
    RetrievalRecall(),
    AnswerRelevancyMetric(model="gpt-4.1-mini"),
    FaithfulnessMetric(model="gpt-4.1-mini"),
    ContextualPrecisionMetric(model="gpt-4.1-mini"),
    ContextualRecallMetric(model="gpt-4.1-mini"),
]

In [25]:
eval = evaluate(
    test_cases=test_cases,
    metrics=metrics,
)

✨ You're running DeepEval's latest Base Metric Metric! (using None, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Base Metric Metric! (using None, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

/Users/manuyadav/Projects/rags_implementations/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 6 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 6 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=53192;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 13.2s | token cost: 0.089664 USD)
» Test Results (16 total tests):
   » Pass Rate: 93.75% | Passed: 15 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Using `rag_eval` package

**NOTE:** Below implementation is of outdated version. Check **rag_2** for the updated implementation.

In [27]:
data = []
with open(test_cases_path, "r") as f:
    # a = f.read()
    for i in f:
        data.append(json.loads(i))

results = evaluate_dataset(data)
results

{'hit@3': 0.9375,
 'precision@3': 0.3125,
 'recall@3': 0.9375,
 'mrr': 0.9375,
 'ndcg@3': 0.9375,
 'failures': [{'question_id': 15,
   'question': 'Which tool is designed to convert PDF documents while preserving their content, and where can its source code be found?',
   'golden_chunk_ids': ['chunk_15'],
   'retrieved_chunk_ids': ['chunk_2'],
   'metrics': {'hit': 0.0,
    'precision': 0.0,
    'recall': 0.0,
    'mrr': 0.0,
    'ndcg': 0.0}}],
 'num_failures': 1}